# Demo C1 — Final Project: Document Research Agent

Capstone project building on `Demo_C1.ipynb` — instead of separate feature demos, this notebook composes several of them into one working agent, built with the current (v1) LangChain/LangGraph APIs. See `CLAUDE.md` at the repo root for the version-specific gotchas already worked out (`create_agent` shape, the `chromadb` pin, `langchain_classic`, etc.).

**Roadmap** — built one stage at a time, each verified before moving to the next:

1. **Knowledge base** — load a PDF, split it, embed it, build a retriever. Tested standalone.
2. **`search_documents` tool** — wrap the retriever as an agent tool. Tested standalone.
3. **Compose tool** — a multi-step LCEL chain (the `RunnablePassthrough.assign` pattern from `Demo_C1`) exposed as a second tool.
4. **Agent, one tool** — wire `search_documents` into `create_agent`.
5. **Agent, both tools** — add the compose tool.
6. **Memory** — add a `checkpointer` so the agent remembers earlier turns.
7. **Structured output** — add a `response_format` Pydantic model.

Only stage 1 is scaffolded below. Later stages get added once this one is working.

## Setup

In [28]:
import os
import warnings

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.storage import LocalFileStore, create_kv_docstore

warnings.filterwarnings("ignore")
load_dotenv()

True

In [29]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [30]:
model = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small", api_key=os.environ.get("OPENAI_API_KEY")
)

PDF_PATH = "../../data/O-príncipe-Nicolau-Maquiavel.pdf"  # same source Demo_C1 already loads
CHROMA_PERSIST_DIR = (
    "../../data/chroma_demo_c1_project"  # persisted so we don't re-embed every restart
)

## 1. Knowledge base

Build the retrieval pipeline from `Demo_C1`'s DOCUMENT / TEXT SPLITTERS / EMBEDDINGS / VECTOR STORES / RETRIEVERS sections, combined into one `ParentDocumentRetriever` (imported from `langchain_classic.retrievers` — see `CLAUDE.md` for why).

You already built this exact pattern in `Demo_C1.ipynb`'s **PARENT DOCUMENT RETRIEVER** section — that's your reference, not this notebook.

Once it runs: test with `retriever.invoke("some query")` and actually look at what comes back — is it a small chunk or the full parent document? That answer matters once this becomes a tool the agent calls, so don't skip inspecting it.

In [31]:
#
# * HOW TO CONSULT FOR EXISTING COLLECTIONS

# // 1 - From an existing vector_store
# vector_store._client.list_collections()

# // 2 - Standalone, without instantiating a langchain Chroma object
# import chromadb

# client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
# client.list_collections()


# * HOW TO DELETE COLLECTIONS

# // 1 - From instantiated Chroma
# vector_store.delete_collection()

# // 2 - From a Standalone
# import chromadb
# client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
# client.delete_collection(name="C1_project")


Once this stage runs and `test_results` looks right to you, let's check in before scaffolding stage 2 (the `search_documents` tool).

### What to measure here ?  
> Did the parent-lookup step fired ?  
>> compare the result against the child:
>>> Close to 400 → you're likely still getting child-level content back, not parents. 
>>>> Meaningfully bigger than 400 → the parent lookup did its job. 

In [32]:
# child_matches[i].page_content → the actual child chunk that matched in Chroma (small, ~400 chars)

# child_matches[i].metadata["doc_id"] → the in-memory id, i.e. the exact key test_results[i] was fetched by from docstore

Implementing the pdf concatenation.

In [33]:
# TODO: load the PDF at PDF_PATH
# HINT: PyPDFLoader(...).load() returns a list[Document]
pdf_docs = PyPDFLoader(PDF_PATH)
pdf_loader = pdf_docs.load()
len(pdf_loader)
pdf_loader[0]

154

Document(metadata={'producer': 'calibre (5.39.1) [https://calibre-ebook.com]', 'creator': 'calibre (5.39.1) [https://calibre-ebook.com]', 'creationdate': '2022-03-13T01:35:03+00:00', 'author': 'Nicolau Maquiavel', 'keywords': 'Political Science, History & Theory', 'moddate': '2022-03-29T14:54:02-03:00', 'title': 'O príncipe', 'source': '../../data/O-príncipe-Nicolau-Maquiavel.pdf', 'total_pages': 154, 'page': 0, 'page_label': '1'}, page_content='')

In [34]:
def concat(docs: list[Document]) -> str:
    return "\n".join(i.page_content for i in docs)


pdf_concat = concat(pdf_loader)
len(pdf_concat)
type(pdf_concat)
pdf_concat[:50]

282670

str

'\nO PRÍNCIPE\nNICOLAU MAQUIAVEL nasceu em Florença e'

In [35]:
new_metadata = {k: v for k, v in pdf_loader[0].metadata.items()}
del new_metadata["page"]
del new_metadata["page_label"]
new_metadata

{'producer': 'calibre (5.39.1) [https://calibre-ebook.com]',
 'creator': 'calibre (5.39.1) [https://calibre-ebook.com]',
 'creationdate': '2022-03-13T01:35:03+00:00',
 'author': 'Nicolau Maquiavel',
 'keywords': 'Political Science, History & Theory',
 'moddate': '2022-03-29T14:54:02-03:00',
 'title': 'O príncipe',
 'source': '../../data/O-príncipe-Nicolau-Maquiavel.pdf',
 'total_pages': 154}

In [36]:
pdf_concat_document = Document(page_content=pdf_concat, metadata=new_metadata)
type(pdf_concat_document)

langchain_core.documents.base.Document

In [37]:
# TODO: define a child_splitter (small chunks, for the vector index) and a
# parent_splitter (larger chunks, for full context) — ParentDocumentRetriever needs both
child_splitter = CharacterTextSplitter(separator="\n", chunk_size=400, chunk_overlap=50)
parent_splitter = CharacterTextSplitter(separator="\n", chunk_size=4000, chunk_overlap=100)

# TODO: create a persisted Chroma vector store (for child chunks) and an
# InMemoryStore (for parent docs), then assemble the ParentDocumentRetriever
# HINT: from langchain_classic.storage import InMemoryStore
# HINT: from langchain_classic.retrievers import ParentDocumentRetriever
# HINT: Chroma(..., persist_directory=CHROMA_PERSIST_DIR, embedding_function=embedding_model)
vector_store = Chroma(
    collection_name="C1_project",
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=embedding_model,
)
docstore = create_kv_docstore(LocalFileStore("../../data/docstore_c1_project"))

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# TODO: index pdf_docs into the retriever
if vector_store._collection.count() == 0:
    retriever.add_documents([pdf_concat_document])
else:
    print(f"Skipping — collection already has {vector_store._collection.count()} entries.")

# TODO: test it — inspect what actually comes back, not just that it ran
test_results = retriever.invoke("qual o nome do livro?")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [38]:
# vector_store.delete_collection()
vector_store._client.list_collections()


# * why is that under vector_store it finds a collection that was instantiated by vector_store_v2 ?
# _client is the underlying chromadb client, and that client is scoped to the persist directory (CHROMA_PERSIST_DIR), not to any single Chroma instance or collection. So, both instances open a client pointed at the same on-disk SQLite/store at CHROMA_PERSIST_DIR.

[Collection(name=C1_project)]

In [39]:
child_matches = vector_store.similarity_search("qual o nome do livro?")

In [40]:
type(child_matches)
len(child_matches)
type(child_matches[0])
len(child_matches[0].page_content)
child_matches[0].page_content

list

4

langchain_core.documents.base.Document

390

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

In [41]:
results_metadata = pd.DataFrame(
    [{"doc_id": i.metadata["doc_id"], "child_page_content": i.page_content} for i in child_matches]
)
results_metadata

,doc_id,child_page_content
0,8bf0bf7d-177e-4bcf-af58-fb7d059b6565,trademarks of Penguin Books Limited and/or Pen...
1,8bf0bf7d-177e-4bcf-af58-fb7d059b6565,REVISÃO\nAna Maria Barbosa\nHuendel V iana\nIS...
2,aa267e52-7eb9-4864-9200-240ba39bd645,XXV. Em que medida a fortuna controla as coisa...
3,aa267e52-7eb9-4864-9200-240ba39bd645,Classics. Faleceu no dia 6 de abril de 2001.\n...


In [42]:
results_metadata["child_page_content"][0]
results_metadata["doc_id"][0]

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

'8bf0bf7d-177e-4bcf-af58-fb7d059b6565'

In [43]:
len(test_results[0].page_content)
test_results[0].page_content

len(child_matches[0].page_content)
child_matches[0].page_content

len(docstore.mget([results_metadata["doc_id"][0]])[0].page_content)
docstore.mget([results_metadata["doc_id"][0]])[0].page_content

1334

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

390

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

1334

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

## 2. `search_documents` tool

Wrap `retriever` (the V2 one, built on `pdf_concat_document`) as an agent tool, using the `@tool` decorator pattern from `Demo_C1`'s **Custom tool via the `@tool` decorator** section — that's your reference, not this notebook.

A few things to think about before you write it:

- The function needs a type-annotated argument (the search query) and a docstring — `@tool` builds the tool's schema and description from those, so the docstring is what the agent reads to decide when to call this.
- `retriever.invoke(query)` returns `list[Document]`. Tools generally return strings back to the agent — so this function has to reduce that list down to something text-based. What's lost if you just join `page_content` with no separator? What do you gain by including each source's metadata alongside its content?
- Per `CLAUDE.md`: tool `name` must match `^[a-zA-Z0-9_-]+$` — no spaces. `@tool` uses the function name by default, so name the function accordingly.

Test it standalone by calling the function directly (not through an agent) before moving on — same rule as stage 1.

In [44]:
# TODO: define search_documents(query: str) -> str
# - call retriever.invoke(query)
# - reduce the resulting list[Document] into a single string to return
# - write a docstring the agent will use to decide when to call this tool


@tool
def search_documents(query: str) -> str:
    """
    Use this to retrieve documents when a question is related to a book called "O príncipe" by Nicolau Maquiavel.
    """
    doc = retriever.invoke(query)
    return "\n".join(i.page_content for i in doc)


In [45]:
# TODO: test standalone — call the underlying function directly (e.g. search_documents.invoke("...")
# or search_documents.func("...") depending on how you call a @tool-wrapped function) and read the output
search_documents.func("o que Maquiavel diz sobre a fortuna e a virtù")

'sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos\narrogante, soube dissimular sua força e seus instintos. Sabia que\ndentre chefes de outras cidades, que também se apressaram a\nvisitar Luís XII buscando alianças, e até mesmo entre seus capitães,\ncorria solta a conspiração contra seu poder . Temiam a aproximação\nde Bórgia com os franceses. Antes mesmo de regressar a Florença,\npouco convencido de conseguir as alianças pretendidas, Maquiavel\npôde assistir aos massacres ordenados por Bórgia para eliminar os\nchefes das outras famílias que buscaram Luís XII. Procedera da\nforma como atuara depois da dominação e pacificação da Romanha\nquando mandou matar seu braço direito, o homem que o havia\najudado a conquistar aquela província para sua glória, Ramirro\nd’Orco, que também fora crudelíssimo na ocupação e era odiado\npelo povo. Orco teve o corpo esquartejado e exposto. Assim, as\ncrueldades da ocupação

## 3. Compose tool

Build a second tool the agent can call — one that doesn't just return raw excerpts like `search_documents`, but composes retrieval *and* generation into a single synthesized answer. Use the multi-step `RunnablePassthrough.assign` pattern from `Demo_C1`'s **LCEL** section (the `location_chain_lcel` → `dish_chain_lcel` → `time_chain_lcel` composition, combined via `overall_chain_lcel`) as your reference — that's your template for "thread new keys through a pipeline," applied here to retrieval + answering instead of location/dish/time.

The shape you're building toward:

1. Start from `{"question": "..."}`.
2. `RunnablePassthrough.assign(context=...)` — call the retriever (or reuse `search_documents.func`) to fetch context for `x["question"]`, adding it under a `"context"` key while keeping `"question"` intact.
3. `RunnablePassthrough.assign(answer=...)` — feed `{"context": ..., "question": ...}` into a `prompt | model | StrOutputParser()` chain to generate the final answer, added under `"answer"`.

Questions worth thinking through before you start:
- What's actually different about what this tool gives the agent, compared to `search_documents`? When might the agent reach for one over the other?
- Your prompt needs `input_variables` matching the keys you assign (`context`, `question`, or whatever names you pick) — a mismatch here fails silently or errors deep in the chain, so keep the naming consistent across cells.

Test each piece standalone — the answer chain alone, then the full composed chain — before wrapping any of it as a `@tool`. Same discipline as stages 1 and 2: isolate failures before they get buried inside an agent's reasoning loop.

In [47]:
# TODO: define a PromptTemplate with input_variables=["context", "question"]
# (or whatever keys you settle on — just stay consistent through the rest of this stage)

template = """ You are a politician adviser on {question}. Responda usando o {context}. YOUR RESPONSE:
"""

prompt = PromptTemplate(input_variables=["question", "context"], template=template)

# TODO: compose the answer chain: prompt | model | StrOutputParser()
answer_chain = prompt | model | StrOutputParser()

In [48]:
# TODO: test answer_chain standalone with a hand-written context — before wiring in retrieval.
# This isolates prompt/model/parser bugs from retriever bugs, testing whether the prompt successfully tells it "answer from this text, not from your own memory." The expected result is a context grounded llm response.

answer_chain.invoke({"question": "como esta o rei?", "context": "o rei esta nu"})


'Como está o rei? O rei está nu. Essa expressão nos lembra que, embora a aparência possa ser enganadora, a verdade muitas vezes é evidente, mesmo que alguns prefiram ignorá-la. É fundamental ter coragem para enfrentar a realidade, por mais desconfortável que possa ser. Assim, encorajamos a transparência e a honestidade na política, para que possamos todos reconhecer e lidar com as verdades que afetam nossa sociedade.'

In [49]:
# TODO: build compose_chain with RunnablePassthrough.assign, two steps:
# 1. assign "context" — call the retriever (or search_documents.func) for x["question"]
# 2. assign "answer"  — call answer_chain.invoke() using the context + question from step 1
# HINT: see Demo_C1's overall_chain_lcel (location -> meal -> recipe -> time) for the exact shape


compose_chain = RunnablePassthrough.assign(
    context=lambda x: search_documents.func(x["question"])
) | RunnablePassthrough.assign(
    answer=lambda x: answer_chain.invoke({"question": x["question"], "context": x["context"]})
)

In [50]:
# TODO: test the full chain: compose_chain.invoke({"question": "..."})
# Check: does result["answer"] actually look grounded in result["context"],
# or does it read like the model ignoring context and answering from general knowledge?
question_1 = "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?"

question_2 = "O que Maquiavel diz sobre o papel da fortuna na vida do príncipe, e ele usa alguma metáfora para isso?"


result = compose_chain.invoke({"question": question_1})
context_1 = result["context"]
# context_2 = result["context"]

In [51]:
result

{'question': "Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?",
 'context': 'sentia mais seguro sobre o que mais convinha à França para conter\nos espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos\narrogante, soube dissimular sua força e seus instintos. Sabia que\ndentre chefes de outras cidades, que também se apressaram a\nvisitar Luís XII buscando alianças, e até mesmo entre seus capitães,\ncorria solta a conspiração contra seu poder . Temiam a aproximação\nde Bórgia com os franceses. Antes mesmo de regressar a Florença,\npouco convencido de conseguir as alianças pretendidas, Maquiavel\npôde assistir aos massacres ordenados por Bórgia para eliminar os\nchefes das outras famílias que buscaram Luís XII. Procedera da\nforma como atuara depois da dominação e pacificação da Romanha\nquando mandou matar seu braço direito, o homem que o havia\najudado a conquistar aquela província para sua glória

In [52]:
print(context_1)
# print(context_2)

sentia mais seguro sobre o que mais convinha à França para conter
os espanhóis ao sul. Mais seguro, mostrou-se mais cordial, menos
arrogante, soube dissimular sua força e seus instintos. Sabia que
dentre chefes de outras cidades, que também se apressaram a
visitar Luís XII buscando alianças, e até mesmo entre seus capitães,
corria solta a conspiração contra seu poder . Temiam a aproximação
de Bórgia com os franceses. Antes mesmo de regressar a Florença,
pouco convencido de conseguir as alianças pretendidas, Maquiavel
pôde assistir aos massacres ordenados por Bórgia para eliminar os
chefes das outras famílias que buscaram Luís XII. Procedera da
forma como atuara depois da dominação e pacificação da Romanha
quando mandou matar seu braço direito, o homem que o havia
ajudado a conquistar aquela província para sua glória, Ramirro
d’Orco, que também fora crudelíssimo na ocupação e era odiado
pelo povo. Orco teve o corpo esquartejado e exposto. Assim, as
crueldades da ocupação feitas por orde

In [53]:
# TODO: wrap compose_chain as a second tool
# - needs a type-annotated query argument and a docstring the agent will read to decide when to call it
# - decide what to return to the agent: just the answer string, or something richer?
# - name it something distinct from search_documents — per CLAUDE.md, name must match ^[a-zA-Z0-9_-]+$


@tool
def get_answer_and_context(query: str) -> dict:
    """Use this to get an answer and the supporting context when a question is related to a book called "O príncipe" by Nicolau Maquiavel."""

    return compose_chain.invoke({"question": query})

In [54]:
# TODO: test standalone (.func(...) or .invoke(...)) before this gets wired into the agent in stage 4
result = get_answer_and_context.func(question_1)

In [55]:
type(result)
result.keys()

dict

dict_keys(['question', 'context', 'answer'])

In [56]:
result["answer"]

'Maquiavel elogia a crueldade de César Bórgia, especialmente suas ações na conquista e pacificação da Romanha, como uma demonstração de eficácia política e pragmatismo na conquista do poder e na manutenção do controle sobre o território. A morte de Ramiro d\'Orco, um líder cru e temido que tinha iluminado a resistência da população contra a dominação, é apresentada por Maquiavel como uma estratégia calculada. Ao eliminar d\'Orco, Bórgia não apenas desmantela um foco de hostilidade em sua administração, mas também se distancia da brutalidade associada à sua autoridade, tornando-se, assim, o governante que traz paz e ordem sob o manto da cruel necessidade. A ação de expor o corpo esquartejado de d\'Orco serve a um duplo propósito: primeiro, elimina um rival potencial que era odiado pelo povo, e segundo, responsabiliza o ex-ministro pela violência, permitindo que Bórgia seja percebido como um príncipe benevolente que simplesmente restaurou a ordem.\n\nNeste contexto, o elogio de Maquiavel

## 4. Agent, one tool

Wire `search_documents` into an agent using `create_agent` — see `Demo_C1`'s **AGENT** section for the reference pattern, and `CLAUDE.md`'s v1 API notes at the repo root for the specifics already worked out for this stack:

- Import from `langchain.agents`, not `langgraph.prebuilt.create_react_agent` (deprecated in `langgraph==1.0`).
- `system_prompt` is a plain `str` (or `SystemMessage`) — no callable/dynamic prompt hook like the old `create_react_agent`'s `prompt` param.
- No memory yet (`checkpointer` is stage 6) and no structured output yet (`response_format` is stage 7) — keep this stage to just: one model, one tool, one system prompt.

Two things worth thinking about before you write the `system_prompt`:
- What should the agent do when a question has nothing to do with the book? (`search_documents`'s docstring already tells the agent *when* to call it — but the system prompt shapes the agent's overall behavior, including when *not* to call any tool.)
- `create_agent` returns a compiled graph, not a plain chain — check `Demo_C1`'s AGENT section for the actual shape of `.invoke(...)`'s input/output (hint: it's message-based, not a plain string in/out like `answer_chain`).

Test by invoking the agent directly with a question you already know `search_documents` handles well (reuse one of your earlier validated questions) — and don't just check the final answer text. Inspect the full message list in the response: did the agent actually *call* the tool, or did it answer from its own knowledge without touching your retriever at all? That distinction matters — an agent that never calls the tool would still produce a plausible-sounding answer, exactly the kind of "looks grounded but isn't" trap you already ran into with `compose_chain`.

In [57]:
# TODO: build the agent
# - model=model
# - tools=[search_documents]
# - system_prompt=... (plain str — decide what it should say about scope/behavior)
agent = create_agent(
    model=model,
    tools=[search_documents],
    system_prompt="""
    You are a research agent for the book called "O Príncipe" by Nicolau Maquiavel. When a question is related to this book, use search_documents to retrieve documents as a context. If the question is not related with this book, answer it by your own knowledge without using the tools.
""",
)

In [58]:
# TODO: invoke the agent with a question you already know search_documents handles well
# HINT: check Demo_C1's AGENT section for the exact input/output shape (message-based,
# not a plain string like answer_chain)
# agent_result = agent.invoke({"messages": [{"role": "user", "content": question_1}]})
agent_result = agent.invoke({"messages": [HumanMessage(content=question_1)]})


In [59]:
agent_result


{'messages': [HumanMessage(content="Por que Maquiavel elogia as ações cruéis de César Bórgia na conquista e pacificação da Romanha, incluindo a morte de Ramiro d'Orco?", additional_kwargs={}, response_metadata={}, id='5172748e-2c0b-45b0-b930-5c93c4028dfa'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 160, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_830d456649', 'id': 'chatcmpl-EE2VeFSoEYRrUMfaerpi07wp4Rtm8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a01264-1d2e-7382-a4eb-b41a873fc84d-0', tool_calls=[{'name': 'search_documents', 'args': {'query': 

In [60]:
# TODO: inspect the full message list in agent_result — don't just read the final answer.
# Look for a ToolMessage (or an AIMessage with tool_calls) confirming search_documents
# actually got invoked, not just a plausible-sounding answer from the model's own knowledge.
